# Assignment 2: Bayesian Generalization

This is the **Python (no-GenJAX)** stencil. The primary path uses `numpy` + `matplotlib`. Optionally, each numpy code cell is followed by a paired *Now in GenJAX* cell that walks through the same task with GenJAX (with enough inline explanation that you can complete them without prior GenJAX experience). The paired cells are not required.

If you would prefer the GenJAX-first stencil, see `generalization.ipynb` in this directory.

For this assignment, you will build a **Bayesian generalization model** for six animals: Cow, Dolphin, Chicken, Seal, Penguin, and Bat. There is no single "right" hypothesis space — *you* design it from properties shared by these animals.

**The setup.** Given that you observe one or more animals have some novel property, how likely is it that the other animals also have it? The Bayesian generalization framework solves this in three steps:

1. **Hypothesis space.** A set $\mathcal{H}$ of hypothesized properties. Each hypothesis $h$ is a binary vector of length 6 (1 if the animal has the property, 0 if not).
2. **Posterior over hypotheses.** After observing animal(s) ${\bf x}$ have the property,
   $$P(h \mid {\bf x}) = \frac{P(h)\,\prod_n P(x_n \mid h)}{\sum_{h' \in \mathcal{H}} P(h')\,\prod_n P(x_n \mid h')}.$$
3. **Predictive distribution over animals.** For each unobserved animal $y$,
   $$P(y \text{ has property} \mid {\bf x}) = \sum_{h:\, y \in h} P(h \mid {\bf x}).$$

We compare two likelihoods:
- **Weak sampling:** $P(x \mid h) = 1$ if $x \in h$, else $0$.
- **Strong sampling:** $P(x \mid h) = 1/|h|$ if $x \in h$, else $0$ (where $|h|$ is the number of animals in $h$).

**Corresponding textbook chapter:** [Tutorial 3 Ch 6 — Generalization](https://josephausterweil.github.io/probintro/intro2/06_generalization/) (and revisit T1 Ch 5 on Bayesian inference).


## Setup

The GenJAX install only matters if you plan to do the optional GenJAX cells. The numpy path works without it.


In [ ]:
# Optional: only needed for the "Now in GenJAX" cells.
# In Google Colab, uncomment on first run:
# !pip install genjax


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import itertools

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

np.random.seed(42)

ANIMALS = ("cow", "dolphin", "chicken", "seal", "penguin", "bat")
N_ANIMALS = len(ANIMALS)
print(f"Animals (in fixed order): {ANIMALS}")


# GenJAX imports — only used by the paired "Now in GenJAX" cells.
# If you skip those, an ImportError here is fine.
# DTYPE note: GenJAX distributions are float32; cast model args with jnp.float32(...) / jnp.int32(...).
try:
    import jax
    import jax.numpy as jnp
    import jax.random as random
    from genjax import gen, categorical
    key = random.PRNGKey(42)
    _GENJAX_AVAILABLE = True
except ImportError:
    _GENJAX_AVAILABLE = False
    print("GenJAX not available — the 'Now in GenJAX' cells will not run. The numpy path works fine.")


---

## Problem 1: Define your hypothesis space

Write down your hypotheses. Each hypothesis is a binary vector of length 6 (one entry per animal in the order Cow, Dolphin, Chicken, Seal, Penguin, Bat). Give each hypothesis a 1–4 word label (e.g. "has wings", "lives in water").

**Constraints:**
- Include a **catch-all** hypothesis containing all six animals.
- Use **more than 4** and **fewer than 63** hypotheses.
- Each entry is 0 or 1 — no animal is "partially" in a hypothesis.

There is no single correct hypothesis space. Pick properties you think are meaningful for these animals.


Represent $\mathcal{H}$ as a numpy array of shape `(H, 6)` — each row is a hypothesis. Keep a parallel list of 1–4 word labels.

In [ ]:
# fill me
#
# Suggested approach:
#   1. List your hypothesis labels (strings, 1-4 words each).
#   2. For each label, write a length-6 binary vector — in the SAME order as
#      ANIMALS = ("cow", "dolphin", "chicken", "seal", "penguin", "bat").
#   3. Stack into a numpy array (dtype=float for math convenience).
#   4. Don't forget the catch-all (all ones).
#   5. Keep `4 < H < 63`.

hypothesis_labels = [
    # your labels here
]

hypothesis_matrix = np.array(
    [
        # your binary rows here
    ],
    dtype=float,
)

H = hypothesis_matrix.shape[0]
assert hypothesis_matrix.shape == (H, N_ANIMALS), f"expected (H, {N_ANIMALS}), got {hypothesis_matrix.shape}"
assert len(hypothesis_labels) == H, "labels and matrix rows must align"
assert 4 < H < 63, "use more than 4 and fewer than 63 hypotheses"
assert (hypothesis_matrix.sum(axis=1) > 0).all(), "no empty hypotheses"
assert (hypothesis_matrix == 1).all(axis=1).any(), "include a catch-all hypothesis"
print(f"H = {H} hypotheses, sizes = {hypothesis_matrix.sum(axis=1).astype(int).tolist()}")


*(1–2 sentences describing the properties you chose.)*

---

## Problem 2: Prior

Define a prior $P(h)$ over your hypotheses. A uniform prior (every hypothesis equally likely) is fine. Write 1–2 sentences justifying your choice.


In [ ]:
# fill me
#
# Suggested approach:
#   1. Uniform prior: np.full(H, 1.0 / H).
#   2. Or any positive length-H vector, normalized.

prior = None   # replace with a length-H numpy array that sums to 1
assert prior is not None and prior.shape == (H,)
assert np.isclose(prior.sum(), 1.0)


*(1–2 sentences justifying your prior.)*

---

## Problem 3: Posterior

Compute the posterior $P(h \mid {\bf x})$ under both **weak** and **strong** sampling. Write the weak/strong likelihood functions, multiply by the prior, normalize.


**Approach.** The hypothesis space is small and discrete, so the posterior is computed by **enumeration**: evaluate the unnormalized posterior at every hypothesis, then normalize.

In [ ]:
# fill me — likelihood under weak and strong sampling
#
# Suggested approach (weak):
#   P(x | h) = 1 if h[x] == 1 else 0.
#   For multiple x's, P(x_1..x_n | h) = prod over n of P(x_n | h) (iid given h).
#
# Suggested approach (strong):
#   P(x | h) = 1/|h| if h[x] == 1 else 0  (where |h| = h.sum()).
#   For multiple x's, prod over n.
#
# Vectorize over hypotheses: pass the full (H, N_ANIMALS) matrix and return a length-H likelihood vector.

def likelihood(hyp_matrix, x_idxs, sampling):
    """Return length-H array of P(x_idxs | h) for each h.

    Args:
        hyp_matrix: shape (H, N_ANIMALS) binary
        x_idxs:     iterable of indices into ANIMALS
        sampling:   "weak" or "strong"
    """
    # fill me
    pass


In [ ]:
# fill me — posterior
#
# Suggested approach:
#   1. unnorm = prior * likelihood(hyp_matrix, x_idxs, sampling)
#   2. post   = unnorm / unnorm.sum()       (assert no division by zero)
#
# Tip: work in log-space if you expect numerical issues. For a small H, linear math is fine.

def posterior(x_idxs, sampling, hyp_matrix=hypothesis_matrix, prior_=prior):
    # fill me
    pass


### 3(a): One observation

Pick one animal and compute the posterior under weak and strong sampling. Plot both as a bar chart over your hypothesis labels.

**Write 1–2 sentences:** how does the posterior change after observing one animal? Are there differences between weak and strong sampling? If so, what are they (and why)?


In [ ]:
# fill me
one_obs_idx = [2]   # e.g. chicken

post_weak_1   = None   # = posterior(one_obs_idx, "weak")
post_strong_1 = None   # = posterior(one_obs_idx, "strong")

# Bar-plot side by side with hypothesis_labels on the x-axis.
# fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
# x = np.arange(H)
# axes[0].bar(x, post_weak_1);   axes[0].set_xticks(x); axes[0].set_xticklabels(hypothesis_labels, rotation=45, ha='right')
# axes[1].bar(x, post_strong_1); ...
# axes[0].set_title("Weak — 1 obs"); axes[1].set_title("Strong — 1 obs")

# your plotting code here


### Now in GenJAX — Problem 3(a) optional

**Concept (Tutorial 2 Ch 2–4): a `@gen` model with a discrete latent.** Here the unknown $h$ is a *discrete* index into the hypothesis matrix, so we use `categorical(logits)`. Unlike continuous latents that require importance sampling, a discrete latent with a small support can be handled by **enumeration**: evaluate the unnormalized posterior at every value of `h_idx` and normalize.

**Key syntax:**
- `h_idx = categorical(log_prior) @ "h_idx"` — sample a discrete index from a categorical with given log-probabilities.
- `model.assess(choices, args)` — return `(log_density, retval)` for a complete trace of choices. We use this to score each hypothesis index.

**Why bother?** For a finite hypothesis space, the numpy enumeration above is faster and more transparent. The GenJAX version below is here to show how the *same* model would look as a `@gen` function — the pattern generalizes to mixed discrete-continuous models where direct enumeration is no longer enough.

In [ ]:
# fill me — optional GenJAX path
#
# Suggested approach:
#   1. Write a @gen function `gen_model(log_prior)` that samples h_idx = categorical(log_prior) @ "h_idx"
#      and returns h_idx.
#   2. For one observation x_1 = chicken (index 2):
#        - Compute the posterior over h_idx by enumeration via `assess`:
#            for each h_idx in 0..H-1, score = log_prior[h_idx] + log_likelihood(hyp_matrix[h_idx], [2], "weak")
#            (model.assess works too but for this simple model the formula above is cleaner)
#        - Normalize to get posterior probabilities.
#      Verify they match the numpy result above.
#
# DTYPE: cast log_prior with jnp.float32(...) before passing it in.

if _GENJAX_AVAILABLE:
    log_prior_j = jnp.log(jnp.array(prior, dtype=jnp.float32))

    @gen
    def gen_model(log_prior):
        # h_idx = categorical(log_prior) @ "h_idx"
        # return h_idx
        pass

    # your enumeration + comparison-to-numpy code here
else:
    print("GenJAX not installed — skipping.")


**Your answer (3a).** *(1–2 sentences.)*

### 3(b): Three observations

Add two more animals (so three animals total) and recompute the posterior under both samplings. Plot.

**Write 1–2 sentences:** how has the posterior changed compared to (a)? How does that differ between weak and strong sampling?


In [ ]:
# fill me
three_obs_idxs = [2, 5, 4]   # e.g. chicken, bat, penguin

post_weak_3   = None
post_strong_3 = None

# your plotting code here


**Your answer (3b).** *(1–2 sentences.)*

---

## Problem 4: Predictive distribution

For each of the six animals $y$, compute
$$P(y \text{ has property} \mid {\bf x}) = \sum_{h:\, y \in h} P(h \mid {\bf x}).$$

Make **four** histograms (1-obs weak, 1-obs strong, 3-obs weak, 3-obs strong) — one bar per animal, height = predictive probability. Label axes and title each plot.

**Write a paragraph** describing the results: which animals are predicted to share the property? How does this differ between weak and strong, and between 1 vs. 3 observations? Tie the patterns back to the hypotheses that survived in the posterior.


In [ ]:
# fill me — predictive distribution
#
# Suggested approach:
#   P(y has property | x) = sum over h where h[y]==1 of posterior[h]
#   This is the matrix product: posterior @ hypothesis_matrix   (shape (H,) @ (H, N_ANIMALS))

def predictive(post, hyp_matrix=hypothesis_matrix):
    # fill me
    pass


# Build the four predictives:
#   pred_weak_1, pred_strong_1, pred_weak_3, pred_strong_3
# Then plot as a 2x2 grid (rows = #obs, cols = sampling).

# your code + plotting here


**Your answer (Problem 4).** *(A paragraph.)*

---

## Problem 5: Break your model

Now expand the hypothesis space to **all** $2^6 - 1 = 63$ non-empty binary vectors of length 6 (i.e. every possible subset of the six animals except the empty set). Use a uniform prior over this expanded space. Recompute the posterior and predictive distributions from Problems 3 and 4.


In [ ]:
# fill me — full 2^6 - 1 hypothesis space
#
# Suggested approach:
#   1. itertools.product([0, 1], repeat=6) -> 64 binary vectors.
#   2. Stack into np.array, drop the all-zeros row (you'll have 63 rows).
#   3. Uniform prior of length 63.
#   4. Reuse posterior() and predictive() — they accept hyp_matrix + prior_ kwargs.

all_hyp_matrix = None    # shape (63, 6)
all_prior      = None    # shape (63,) uniform

# your code here


In [ ]:
# fill me — recompute and plot.
#
# Recompute the 1-obs and 3-obs posteriors + predictives under the expanded space.
# You don't need to dump all 63 posterior values — show the predictive (which is length 6).

# your code + plotting here


**Write a few sentences:** what happened to the posterior and predictive probabilities? Why? **Relate this to a theorem we covered in class.** (Hint: think about what a uniform prior over *all possible* hypotheses encodes about your beliefs — and what it does *not* encode.)

You do not need to dump all 63 hypotheses' probabilities in your report — just enough to make your point.


**Your answer (Problem 5).** *(A few sentences.)*

---

## Submission

Submit by DM or email to the instructor **one** of:

- your completed notebook — it must run end-to-end with no errors and contain your figures, inline text answers, and descriptions; **or**
- a single PDF report containing your code, figures, text answers, and descriptions.
